In [ ]:
%pip install pandas requests fredapi matplotlib python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# ENTSOG Weekly Country Panel

Builds weekly country panels for DE, FR, IT, NL, ES, UK and an EU aggregate from ENTSOG AggregatedData.

Pipeline behavior:
- First run: backfill from 2018-01-01
- Later runs: incremental fetch from the last stored date with overlap buffer
- Primary indicators: Physical Flow and Allocation
- Units: convert daily kWh/d values into weekly TWh
- Derived metrics: level, WoW %, YoY %

## Setup

**FRED API Key required:** Create a `.env` file in the workspace root by copying `.env.example`:
```bash
cp .env.example .env
```
Then edit `.env` and add your FRED API key from https://fred.stlouisfed.org/docs/api/

In [ ]:
from __future__ import annotations

import os
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

# -----------------------------
# Config
# -----------------------------
BASE_URL = "https://transparency.entsog.eu/api/v1/AggregatedData"
COUNTRIES = ["DE", "FR", "IT", "NL", "ES", "UK"]
INDICATORS = ["Physical Flow", "Allocation"]

START_DATE = date(2018, 1, 1)
FORCE_MAX_HISTORY_PULL = True
OVERLAP_DAYS = 7
TIMEZONE = "CET"
PERIOD_TYPE = "day"
PAGE_LIMIT = 1000
MAX_RETRIES = 4
RETRY_BACKOFF_SECONDS = 1.5
REQUEST_TIMEOUT_SECONDS = 60

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "methodo" else Path.cwd().resolve()
OUTPUT_DIR = ROOT / "data" / "processed"
DAILY_OUTPUT = OUTPUT_DIR / "entsog_daily_selected.csv"
WEEKLY_OUTPUT = OUTPUT_DIR / "entsog_weekly_panel.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Daily output: {DAILY_OUTPUT}")
print(f"Weekly output: {WEEKLY_OUTPUT}")
print(f"Force max-history pull: {FORCE_MAX_HISTORY_PULL}")

Root: /workspaces/high_frequency
Daily output: /workspaces/high_frequency/data/processed/entsog_daily_selected.csv
Weekly output: /workspaces/high_frequency/data/processed/entsog_weekly_panel.csv
Force max-history pull: True


In [ ]:
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# Load environment variables from .env file
ENV_FILE = ROOT.parent / ".env"
if ENV_FILE.exists():
    load_dotenv(ENV_FILE)
    print(f"✓ Loaded .env file from {ENV_FILE}")
else:
    print(f"⚠ .env file not found at {ENV_FILE}")
    print(f"  Create a .env file by copying .env.example and filling in your FRED_API_KEY")

In [29]:
def _safe_get_json(session: requests.Session, url: str, params: dict) -> dict:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

            # ENTSOG returns 404 JSON messages for "No result found" and archive limits.
            if resp.status_code == 404:
                return {}

            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            last_err = exc
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
            else:
                raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts. params={params}") from last_err


def _month_start(d: date) -> date:
    return date(d.year, d.month, 1)


def _next_month(d: date) -> date:
    if d.month == 12:
        return date(d.year + 1, 1, 1)
    return date(d.year, d.month + 1, 1)


def _resolve_available_start_date(
    session: requests.Session,
    requested_start: date,
    end_date: date,
    probe_indicator: str,
) -> date:
    # Probe monthly windows and return the first month that yields data.
    probe = _month_start(requested_start)
    while probe <= end_date:
        nxt = _next_month(probe)
        window_end = min(nxt - timedelta(days=1), end_date)
        params = {
            "from": probe.isoformat(),
            "to": window_end.isoformat(),
            "indicator": probe_indicator,
            "periodType": PERIOD_TYPE,
            "timeZone": TIMEZONE,
            "limit": PAGE_LIMIT,
            "offset": 0,
        }
        payload = _safe_get_json(session, BASE_URL, params)
        chunk = payload.get("AggregatedData", payload.get("aggregatedData", []))
        if chunk:
            return probe
        probe = nxt

    return requested_start


def fetch_aggregated_data(
    start_date: date,
    end_date: date,
    countries: list[str],
    indicators: list[str],
) -> pd.DataFrame:
    rows: list[dict] = []
    session = requests.Session()

    effective_start = _resolve_available_start_date(
        session=session,
        requested_start=start_date,
        end_date=end_date,
        probe_indicator=indicators[0],
    )

    print(f"Effective historical start used by API: {effective_start}")

    # Pull month-by-month to avoid sparse/partial results from huge windows.
    for indicator in indicators:
        cursor = _month_start(effective_start)
        while cursor <= end_date:
            nxt = _next_month(cursor)
            window_end = min(nxt - timedelta(days=1), end_date)

            offset = 0
            while True:
                params = {
                    "from": cursor.isoformat(),
                    "to": window_end.isoformat(),
                    "indicator": indicator,
                    "periodType": PERIOD_TYPE,
                    "timeZone": TIMEZONE,
                    "limit": PAGE_LIMIT,
                    "offset": offset,
                }

                payload = _safe_get_json(session, BASE_URL, params)
                chunk = payload.get("AggregatedData", payload.get("aggregatedData", []))
                if not chunk:
                    break

                rows.extend(chunk)
                if len(chunk) < PAGE_LIMIT:
                    break

                offset += PAGE_LIMIT

            cursor = nxt

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

In [ ]:
# -----------------------------
# Incremental date window
# -----------------------------
today = date.today()

if DAILY_OUTPUT.exists():
    existing_daily = pd.read_csv(DAILY_OUTPUT)
else:
    existing_daily = pd.DataFrame()

if FORCE_MAX_HISTORY_PULL:
    last_date = pd.to_datetime(existing_daily["date"], errors="coerce").dt.date.max() if (not existing_daily.empty and "date" in existing_daily.columns) else None
    fetch_start = START_DATE
else:
    if not existing_daily.empty and "date" in existing_daily.columns:
        existing_daily["date"] = pd.to_datetime(existing_daily["date"], errors="coerce").dt.date
        last_date = existing_daily["date"].max()
        fetch_start = max(START_DATE, last_date - timedelta(days=OVERLAP_DAYS))
    else:
        last_date = None
        fetch_start = START_DATE

print(f"Last stored date: {last_date}")
print(f"Fetch window: {fetch_start} -> {today}")

new_raw = fetch_aggregated_data(fetch_start, today, COUNTRIES, INDICATORS)
print(f"Fetched rows: {len(new_raw):,}")

if new_raw.empty and existing_daily.empty:
    raise RuntimeError("No data fetched and no existing local data available.")

if not new_raw.empty:
    keep_cols = [
        "id",
        "countryKey",
        "countryLabel",
        "indicator",
        "periodType",
        "periodFrom",
        "periodTo",
        "unit",
        "value",
        "flowStatus",
        "lastUpdateDateTime",
        "directionKey",
        "adjacentSystemsLabel",
        "operatorLabel",
        "bzShort",
    ]
    existing_cols = [c for c in keep_cols if c in new_raw.columns]
    new_daily = new_raw[existing_cols].copy()

    new_daily["date"] = pd.to_datetime(new_daily["periodFrom"], errors="coerce", utc=True).dt.date
    new_daily["value"] = pd.to_numeric(new_daily["value"], errors="coerce")
    new_daily["lastUpdateDateTime"] = pd.to_datetime(new_daily["lastUpdateDateTime"], errors="coerce", utc=True)

    new_daily = new_daily[new_daily["countryKey"].isin(COUNTRIES)]
    new_daily = new_daily[new_daily["indicator"].isin(INDICATORS)]
    new_daily = new_daily[new_daily["value"].notna()]
else:
    new_daily = pd.DataFrame()

if existing_daily.empty:
    combined_daily = new_daily.copy()
else:
    combined_daily = pd.concat([existing_daily, new_daily], ignore_index=True, sort=False)

if not combined_daily.empty:
    combined_daily["lastUpdateDateTime"] = pd.to_datetime(combined_daily["lastUpdateDateTime"], errors="coerce", utc=True)
    combined_daily = combined_daily.sort_values(["id", "lastUpdateDateTime"])
    # Keep the latest revision per ENTSOG id
    if "id" in combined_daily.columns:
        combined_daily = combined_daily.drop_duplicates(subset=["id"], keep="last")

# Compute EU daily aggregate (sum across 6 countries per indicator per day)
eu_daily = (
    combined_daily[combined_daily["countryKey"].isin(COUNTRIES)]
    .groupby(["date", "indicator", "periodType", "unit"], as_index=False)["value"]
    .sum()
    .assign(
        countryKey="EU",
        countryLabel="EU Aggregate",
        id=None,
        flowStatus=None,
        lastUpdateDateTime=None,
        directionKey=None,
        adjacentSystemsLabel=None,
        operatorLabel=None,
        bzShort=None,
        periodFrom=None,
        periodTo=None,
    )
)

# Reorder columns to match combined_daily
eu_daily = eu_daily[combined_daily.columns]

# Append EU aggregate to combined daily
combined_daily = pd.concat([combined_daily, eu_daily], ignore_index=True, sort=False)
combined_daily = combined_daily.sort_values(["date", "countryKey", "indicator"]).reset_index(drop=True)

combined_daily.to_csv(DAILY_OUTPUT, index=False)
print(f"Saved daily dataset: {len(combined_daily):,} rows (including EU aggregate)")
print(f"Countries in daily output: {sorted(combined_daily['countryKey'].dropna().unique().tolist())}")

Last stored date: 2026-04-29
Fetch window: 2018-01-01 -> 2026-04-29
Effective historical start used by API: 2019-11-01
Fetched rows: 104,281
Saved daily dataset: 47,697 rows


In [31]:
# -----------------------------
# Weekly panel construction
# -----------------------------
DROP_INCOMPLETE_LAST_WEEK = True


daily = combined_daily.copy()
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily = daily.dropna(subset=["date", "value", "countryKey", "indicator"])

# Monday-start week key
daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.weekday, unit="D")
daily["twh"] = daily["value"] / 1_000_000_000.0

country_weekly = (
    daily.groupby(["countryKey", "indicator", "week_start"], as_index=False)["twh"]
    .sum()
    .rename(columns={"twh": "twh_week"})
)

# Reindex to contiguous weekly frequency per country/indicator,
# so WoW/YoY do not compare across multi-year gaps.
parts = []
for (country, indicator), g in country_weekly.groupby(["countryKey", "indicator"], as_index=False):
    g = g.sort_values("week_start")
    full_weeks = pd.date_range(g["week_start"].min(), g["week_start"].max(), freq="W-MON")
    gg = g.set_index("week_start").reindex(full_weeks)
    gg.index.name = "week_start"
    gg = gg.reset_index()
    gg["countryKey"] = country
    gg["indicator"] = indicator
    parts.append(gg[["countryKey", "indicator", "week_start", "twh_week"]])

country_weekly = pd.concat(parts, ignore_index=True)

if DROP_INCOMPLETE_LAST_WEEK:
    current_week_start = pd.Timestamp.today().normalize() - pd.to_timedelta(pd.Timestamp.today().weekday(), unit="D")
    country_weekly = country_weekly[country_weekly["week_start"] < current_week_start]

eu_weekly = (
    country_weekly.groupby(["indicator", "week_start"], as_index=False)["twh_week"]
    .sum(min_count=1)
    .assign(countryKey="EU")
)
eu_weekly = eu_weekly[["countryKey", "indicator", "week_start", "twh_week"]]

panel = pd.concat([country_weekly, eu_weekly], ignore_index=True)
panel = panel.sort_values(["countryKey", "indicator", "week_start"]).reset_index(drop=True)

panel["wow_pct"] = panel.groupby(["countryKey", "indicator"])["twh_week"].pct_change(1)
panel["yoy_pct"] = panel.groupby(["countryKey", "indicator"])["twh_week"].pct_change(52)

panel.to_csv(WEEKLY_OUTPUT, index=False)

print(f"Saved weekly panel: {len(panel):,} rows")
print("Countries in output:", sorted(panel["countryKey"].dropna().unique().tolist()))
print("Indicators in output:", sorted(panel["indicator"].dropna().unique().tolist()))

panel.tail(20)

Saved weekly panel: 4,713 rows
Countries in output: ['DE', 'ES', 'EU', 'FR', 'IT', 'NL', 'UK']
Indicators in output: ['Allocation', 'Physical Flow']


,countryKey,indicator,week_start,twh_week,wow_pct,yoy_pct
4693,UK,Physical Flow,2025-12-08,NaN,NaN,NaN
4694,UK,Physical Flow,2025-12-15,NaN,NaN,NaN
4695,UK,Physical Flow,2025-12-22,NaN,NaN,NaN
4696,UK,Physical Flow,2025-12-29,11.754276,NaN,1.123306
4697,UK,Physical Flow,2026-01-05,NaN,NaN,NaN
4698,UK,Physical Flow,2026-01-12,NaN,NaN,NaN
4699,UK,Physical Flow,2026-01-19,NaN,NaN,NaN
4700,UK,Physical Flow,2026-01-26,5.535827,NaN,-0.080135
4701,UK,Physical Flow,2026-02-02,5.641220,0.019038,NaN
4702,UK,Physical Flow,2026-02-09,NaN,NaN,NaN


# FRED Energy Prices: Brent & European Natural Gas

Fetches maximum-history Brent crude oil (Europe) and European natural gas price data from the Federal Reserve Economic Data (FRED) API.

**Series:**
- MCOILBRENTEU: Brent Europe (USD/bbl)
- PNGASEUUSDM: Natural Gas Europe (USD/MMBtu)

**Coverage:** As much history as FRED provides (typically 1980s onwards), monthly frequency

In [33]:
import fredapi

# Get FRED API key from environment; raise if not set
FRED_API_KEY = os.getenv("FRED_API_KEY")
if not FRED_API_KEY:
    raise RuntimeError("FRED_API_KEY environment variable not set. Set it before running this notebook.")

# Initialize FRED client
fred = fredapi.Fred(api_key=FRED_API_KEY)

# Target FRED series
FRED_SERIES_CONFIG = {
    "MCOILBRENTEU": {"name": "Brent Europe", "unit": "USD/bbl"},
    "PNGASEUUSDM": {"name": "Natural Gas Europe", "unit": "USD/MMBtu"},
}

FRED_START_DATE = date(1980, 1, 1)  # Request from 1980; FRED returns earliest available
FRED_OUTPUT = OUTPUT_DIR / "fred_energy_prices.csv"

print(f"FRED output file: {FRED_OUTPUT}")
print(f"Series to fetch: {list(FRED_SERIES_CONFIG.keys())}")

ModuleNotFoundError: No module named 'fredapi'

In [ ]:
# Fetch data from FRED for each series
fred_data = {}
for series_id, series_info in FRED_SERIES_CONFIG.items():
    print(f"\nFetching {series_info['name']} ({series_id})...")
    try:
        # fredapi.get_series() returns all available data
        series = fred.get_series(
            series_id,
            observation_start=FRED_START_DATE,
        )
        fred_data[series_id] = series
        print(f"  ✓ Fetched {len(series):,} observations")
        print(f"    Date range: {series.index.min()} to {series.index.max()}")
    except Exception as e:
        print(f"  ✗ Error fetching {series_id}: {e}")

print(f"\nTotal series fetched: {len(fred_data)}/{len(FRED_SERIES_CONFIG)}")

In [ ]:
# Standardize time index and missing values
cleaned_fred_series = {}
for series_id, series in fred_data.items():
    # Ensure datetime index
    series.index = pd.to_datetime(series.index)
    
    # Coerce values to numeric
    series = pd.to_numeric(series, errors="coerce")
    
    # Sort ascending by date
    series = series.sort_index()
    
    # Record info
    null_count = series.isna().sum()
    cleaned_fred_series[series_id] = series
    
    print(f"\n{FRED_SERIES_CONFIG[series_id]['name']} ({series_id}):")
    print(f"  Date range: {series.index.min()} to {series.index.max()}")
    print(f"  Observations: {len(series):,}")
    print(f"  Missing values: {null_count:,} ({100.0 * null_count / len(series):.2f}%)")
    print(f"  Min: {series.min():.2f}, Max: {series.max():.2f}, Mean: {series.mean():.2f}")

In [ ]:
# Build combined energy prices DataFrame
df_fred = pd.DataFrame()

# Add each series with a clear column name
df_fred["brent_europe_usd_bbl"] = cleaned_fred_series.get("MCOILBRENTEU")
df_fred["natgas_europe_usd_mmbtu"] = cleaned_fred_series.get("PNGASEUUSDM")

# Reset index to have date as a column
df_fred = df_fred.reset_index()
df_fred.rename(columns={"index": "date"}, inplace=True)

print(f"Combined FRED DataFrame shape: {df_fred.shape}")
print(f"Date range: {df_fred['date'].min()} to {df_fred['date'].max()}")
print(f"\nColumns: {df_fred.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df_fred.head()}")
print(f"\nLast 5 rows:\n{df_fred.tail()}")

In [ ]:
# Quality checks and range validation
print("="*60)
print("FRED DATA QUALITY CHECKS")
print("="*60)

print("\nNull counts per column:")
print(df_fred.isnull().sum())

print("\nDuplicate dates:")
dup_dates = df_fred[df_fred.duplicated(subset=["date"], keep=False)]
if len(dup_dates) > 0:
    print(f"  Found {len(dup_dates)} duplicate dates")
    print(dup_dates)
else:
    print("  None")

print("\nSummary statistics:")
print(df_fred[["brent_europe_usd_bbl", "natgas_europe_usd_mmbtu"]].describe())

# Check overlapping window
brent_dates = df_fred[df_fred["brent_europe_usd_bbl"].notna()]["date"]
natgas_dates = df_fred[df_fred["natgas_europe_usd_mmbtu"].notna()]["date"]

if len(brent_dates) > 0 and len(natgas_dates) > 0:
    overlap_start = max(brent_dates.min(), natgas_dates.min())
    overlap_end = min(brent_dates.max(), natgas_dates.max())
    print(f"\nJoint sample window (both series available):")
    print(f"  Start: {overlap_start}, End: {overlap_end}")
    overlap_count = len(df_fred[(df_fred["date"] >= overlap_start) & 
                                 (df_fred["date"] <= overlap_end) &
                                 (df_fred["brent_europe_usd_bbl"].notna()) &
                                 (df_fred["natgas_europe_usd_mmbtu"].notna())])
    print(f"  Observations with both series: {overlap_count}")
else:
    print("\nWarning: No overlapping sample window.")

In [ ]:
# Visualize Brent vs European Natural Gas
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Raw prices
ax1 = axes[0]
ax1_twin = ax1.twinx()

ax1.plot(df_fred["date"], df_fred["brent_europe_usd_bbl"], 
         label="Brent Europe (USD/bbl)", color="blue", linewidth=1.5)
ax1_twin.plot(df_fred["date"], df_fred["natgas_europe_usd_mmbtu"], 
              label="Natural Gas Europe (USD/MMBtu)", color="red", linewidth=1.5)

ax1.set_xlabel("Date")
ax1.set_ylabel("Brent Europe (USD/bbl)", color="blue")
ax1_twin.set_ylabel("Natural Gas Europe (USD/MMBtu)", color="red")
ax1.tick_params(axis="y", labelcolor="blue")
ax1_twin.tick_params(axis="y", labelcolor="red")
ax1.grid(True, alpha=0.3)
ax1.set_title("FRED Energy Prices: Raw Series (MCOILBRENTEU, PNGASEUUSDM)")
ax1.legend(loc="upper left")
ax1_twin.legend(loc="upper right")

# Plot 2: Normalized (z-score) for comparability
ax2 = axes[1]

brent_normalized = (df_fred["brent_europe_usd_bbl"] - df_fred["brent_europe_usd_bbl"].mean()) / df_fred["brent_europe_usd_bbl"].std()
natgas_normalized = (df_fred["natgas_europe_usd_mmbtu"] - df_fred["natgas_europe_usd_mmbtu"].mean()) / df_fred["natgas_europe_usd_mmbtu"].std()

ax2.plot(df_fred["date"], brent_normalized, label="Brent Europe (normalized)", color="blue", linewidth=1.5)
ax2.plot(df_fred["date"], natgas_normalized, label="Natural Gas Europe (normalized)", color="red", linewidth=1.5)
ax2.axhline(y=0, color="black", linestyle="--", alpha=0.3)
ax2.set_xlabel("Date")
ax2.set_ylabel("Normalized Price (z-score)")
ax2.grid(True, alpha=0.3)
ax2.set_title("FRED Energy Prices: Normalized Series (z-score)")
ax2.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fred_energy_prices_plot.png", dpi=100, bbox_inches="tight")
plt.show()

print("✓ Plot saved to data/processed/fred_energy_prices_plot.png")

In [ ]:
# Export clean FRED dataset for downstream use
df_fred.to_csv(FRED_OUTPUT, index=False)
print(f"✓ Saved FRED energy prices to {FRED_OUTPUT}")

# Sanity check: reload and verify
df_fred_check = pd.read_csv(FRED_OUTPUT)
print(f"\nReload verification:")
print(f"  Shape: {df_fred_check.shape}")
print(f"  Columns: {df_fred_check.columns.tolist()}")
print(f"  Date range: {df_fred_check['date'].min()} to {df_fred_check['date'].max()}")
print(f"  Null counts:\n{df_fred_check.isnull().sum()}")
print(f"\nFirst 3 rows:\n{df_fred_check.head(3)}")
print(f"\nLast 3 rows:\n{df_fred_check.tail(3)}")
print("\n✓ FRED dataset ready for downstream modeling.")